# Simple Climate Model

# Part 1: Comparing Linearized and Non-Linear Simulations

## Introduction

This script adapts a simple climate model from https://www.e-education.psu.edu/meteo469/node/198
and uses it to illustrate linearization and discrete-time simulation of a
nonlinear dynamical system.

The one layer energy model of the climate has two temperature nodes one for the atmosphere temperature and one for the earth surface temperature. By the virtue of energy balance at each of these nodes the earth surface temperature can be expressed using the following relationship.

$$ \frac{dT}{dt} = \frac{\pi R^2}{C}[(1-\alpha)S - 4\sigma(1-\epsilon/2)T^4] \tag{1}$$

where $T$ is the surface temperature, $R$ is the radius of the earth, $C$ is the total heat capacity of earth, $\alpha$ is the solar radiation absorbtivity (or emissivity) of the upper atmosphere also referred to as Earth's *albedo*, $S$ is the incident solar radiation or solar constant, $\epsilon$ is the emissivity (or absorbitivity) of the atmosphere to long-wave radiation emitted by the earth's surface, and $\sigma$ is the Stefan-Boltzmann's constant. Here, $\epsilon$ is a function of time and it depends on the $CO_2$ concentration in the atmosphere. Increasing concentration of $CO_2$ in the atmosphere gives rise to an increased value of $\epsilon$ due to the Greenhouse effect.

Eqn. $(1)$ represents a non-linear first order dynamical system and can be written in the cannonical form of eqn. $(2)$ as shown in eqn. $(3)$

$$ \frac{dx(t)}{dt} = f(x(t),u(t),\tilde{w}(t)) \tag{2}$$

$$ \frac{dx(t)}{dt} = f(x(t),u(t),\tilde{w}(t)) = -\beta(1-u(t)/2)x(t)^4 + \tilde{w}(t) \tag{3} $$

where, the state variable $x(t) = T(t)$, the control/action variable $u(t) = \epsilon(t)$, the continuous-time disturbance $\tilde{w}(t) = \pi R^2(1-\alpha(t))S/C$, and the parameter $\beta = 4\sigma\pi R^2/C$. Eqn. $(3)$ can be time discretized and solved as an initial value problem (IVP) using classical numerical solvers given nominal values of $\hat{u}(t)$ and $\hat{\tilde{w}}(t)$. On the other hand, it can also be linearized in a piece-wise fashion and also solved as an IVP. The objective of this notebook is two implement and compare both the approaches.

## Nonlinear Solution

In order to solve the problem using an in-built ODE solver function is created that will take the initial global surface temperature values, the $\epsilon(t)$ values as the control input $u(t)$, the continuous time disturbance $\tilde{w}(t)$, and the parameter $\beta$.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp


def nonlinear_climate_sim(t, x0, u, wt, beta):
    """
    Simulates nonlinear climate dynamics.

    Args:
        t (array-like): Time span in seconds.
        x0 (float): Initial global average surface temperature in K.
        u (np.ndarray): K vector of atmospheric emissivities.
        wt (np.ndarray): K vector of continuous-time disturbances in K/s.
        beta (float): Parameter in K^3/s.

    Returns:
        x: K+1 vector of global average surface temperatures in K.
    """
    K = len(u)  # Number of time steps
    # Ensure input is a 1D vector
    if u.ndim != 1 or wt.ndim != 1:
        raise ValueError("u and wt must be 1D arrays (vectors).")

    # Initialize output array
    x = np.zeros(K + 1)  #  K+1 vector for global average surface temperatures in K
    x[0] = x0  # Initial state

    # Perform the simulation

    for k in range(K):
        #
        # Your
        #
        # code
        #
        # here
        #

    return x

IndentationError: expected an indented block after 'for' statement on line 30 (677381898.py, line 39)

## Linearized Solution

Recall from theory that eqn. $(1)$ can be linearized around $\hat{x}(t)$, $\hat{u}(t)$,and $\hat{w}(t)$ using Taylor's theorem as,

 $$\frac{d\delta_x(t)}{dt} = \tilde{A}(t)\delta_x(t) + \tilde{B}(t)\delta_u(t) + \tilde{G}(t)\delta_w(t) \tag{4}$$

 where $\delta_x(t), \delta_u(t), \delta_w(t)$ are defined as perturbation around the nominal values of $\hat{x}(t)$, $\hat{u}(t)$,and $\hat{w}(t)$ given by $\delta_x(t) = x(t) - \hat{x}(t), \delta_u(t) = u(t) - \hat{u}(t), \delta_w(t) = w(t) - \hat{w}(t)$. The values of $\tilde{A}(t), \tilde{B}(t), \& \tilde{G}(t)$ are defined as,

$$ \tilde{A}(t) = \left. \frac{\partial f}{\partial x} \right|_{\hat{x}(t), \hat{u}(t), \hat{w}(t)} \tag{5}$$
$$ \tilde{B}(t) = \left. \frac{\partial f}{\partial u} \right|_{\hat{x}(t), \hat{u}(t), \hat{w}(t)} \tag{6}$$
$$ \tilde{G}(t) = \left. \frac{\partial f}{\partial w} \right|_{\hat{x}(t), \hat{u}(t), \hat{w}(t)} \tag{7}$$

Notice that eqn. $(4)$ now is in the form of a linear dynamical system (LDS) with $\delta_x(t)$ as the state variable, $\delta_u(t)$ as the action variable, and $G\delta_w(t)$ as the disturbance. Based on $f(x,u,w)$ defined in eqn. $(3)$, $\tilde{A}(t), \tilde{B}(t), \& \tilde{G}(t)$ can be calculated as,

$$ f(t) = \left(-\beta(1-u(t)/2)x(t)^4 + \tilde{w}(t)\right) \tag{8}$$

$$\tilde{A}(t) = \frac{\partial f}{\partial x} = -4\beta(1-u(t)/2)x(t)^3 \tag{9}$$

$$\tilde{B}(t) = \frac{\partial f}{\partial u}  = \beta x(t)^4/2 \tag{10}$$

$$\tilde{G}(t) = \frac{\partial f}{\partial w}  = 1 \tag{11}$$

In order to solve this LDS, it first needs to be discretized. Consider an LDS problem shown in shown in eqn. $(12)$

$$\frac{dx(t)}{dt} = \tilde{A}(t)x(t) + \tilde{B}(t)u(t) + \tilde{w}(t) \tag{12}$$

The discretized form of this equation is given by,

$$x(k+1) = A(k)x(k) + B(k)u(k) + w(k) \tag{13}$$

where $k$ denotes $t_k$ and $A,B,w$ are represented by,

$$A(t_k) = e^{(t_{k+1} - t_k)\tilde{A}(t_k)} \tag{14}$$

$$ B(t_k) = e^{t_{k+1} \tilde{A}(t_k)} \int_{t_k}^{t_{k+1}} e^{-\tau \tilde{A}(t_k)} \ d\tau \ \tilde{B}(t_k) \tag{15}$$

$$ w(t_k) = e^{t_{k+1} \tilde{A}(t_k)} \int_{t_k}^{t_{k+1}} e^{-\tau \tilde{A}(t_k)} \ d\tau \ \tilde{w}(t_k) \tag{16}$$

Using this derivation, we discretize the linear dynamic system developed in eqn. $(4)$. Assuming a uniform timestep and assuming that within that timestep $\tilde{A}(t), \tilde{B}(t), \delta_u(t), \& \delta_{\tilde{w}}(t)$ are constants, eqn. $(4)$ can be discretized as,

$$\delta_x(t_k) = A(t_k)\delta_x(t_k) + B(t_k)\delta_u(t_k) + \delta_w(t_k) \tag{17}$$

$$A(t_k) = e^{\Delta t \tilde{A}(t_k)} \tag{18}$$

$$ B(t_k) = \frac{A(t_k)-1}{\tilde{A}(t_k)}\tilde{B}(t_k) \tag{19} $$

$$ \delta_w(t_k) = \frac{A(t_k)-1}{\tilde{A}(t_k)}\delta_{\tilde{w}}(t_k) \tag{20}$$

Replace $\tilde{A}(t_k)$ and $\tilde{B}(t_k)$ from equation $9$ and $10$ evaluated at the discretized time interval $t_k$ and substituted in eqn. $18$, $19$ and $20$.

In [ ]:
def linearized_climate_sim(t, x_hat, u_hat, du, dwt, beta):
    """
    Simulates linearized climate dynamics.

    Args:
        t (array-like): Time span in seconds.
        x_hat (np.ndarray): K+1 vector of nominal global average surface temperatures in K.
        u_hat (np.ndarray): K vector of nominal atmospheric emissivities.
        du (np.ndarray): K vector of atmospheric emissivity perturbations.
        dwt (np.ndarray): K vector of continuous-time disturbance perturbations in K/s.
        beta (float): Parameter in K^3/s.

    Returns:
        x_lin: K+1 vector of approximate global average surface temperatures in K.
    """
    K = len(u_hat)  # Number of time steps
    dx = np.zeros_like(x_hat)  # global average surface temperature perturbations in K

    for k in range(K):
       #
        # Your
        #
        # code
        #
        # here
        #

    return x_lin

## Analysis

Now that we have developed the models for solving the simple climate equation in a linearized and discretized way and the non-linear way we will now compare the two solutions.

First let us define some standard imports and constants.

In [ ]:
def k2c(k):
    """
    Converts Kelvin to Celsius.
    Args:
        k (float or array-like): Temperature in Kelvin.
    Returns:
        float or array-like: Temperature in Celsius.
    """
    return k - 273

# ==============================================================================
# Required imports
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt

# ==============================================================================
# graphics settings
# ==============================================================================

plt.rc('font', size=15)  # Default font size for all text
plt.rc('axes', titlesize=15, labelsize=15)  # Font size for axes titles and labels
plt.rc('xtick', labelsize=15)  # Font size for x-axis tick labels
plt.rc('ytick', labelsize=15)  # Font size for y-axis tick labels
plt.rc('legend', fontsize=15)  # Font size for legend
plt.rc('figure', titlesize=20)  # Font size for figure titles

# ==============================================================================
# Input Data
# ==============================================================================

alpha = 0.3  # Albedo of Earth's atmosphere
S = 1366  # Solar constant, W/m^2
eps = 0.767  # Emissivity of Earth's atmosphere
sigma = 5.67e-8  # Stefan-Boltzmann constant, W/m^2/K^4
rho = 997  # Density of water, kg/m^3
c = 4.186e3  # Specific heat of water, J/kg/K
R = 6.378e6  # Radius of Earth, m
l = 70  # Depth of well-mixed water layer on Earth's surface, m

C = 0.7 * 4 * np.pi * R**2 * rho * c * l  # Thermal capacitance of Earth's surface, J/K
beta = 4 * sigma * np.pi * R**2 / C  # Intermediate coefficient, K^3/s

To solve this problem we will need to provide a few inputs. In addition to the values of several constants that will be used to solve this problem, the concentration of $CO_2$ which affects the $\epsilon$ of the lower atmosphere to long wave radiation is also an input. Recall that the change in emissivity drives the change in surface temperature and hence is the control/action variable. The emissivity is written as linear function of the $CO_2$ concentration as given by,

$$\epsilon = \epsilon_0 + m*C_{CO_2} \tag{21}$$

where $\epsilon_0$ is the original emissivity value which will be assumed to be the average value from 1880-1900, $m$ is the slope which will be calculated based on a linear interpolation between two known values of temperature and concentration and $C_{CO_2}$ is the value of concentration of $CO_2$ mesured in *parts per million (ppm)*. Recall from the lecture that the average steady state temperature can be calculated as,

$$T = \sqrt[4]{\frac{(1-\alpha) S}{4\sigma (1-\varepsilon/2)}} \tag{22} $$
Therefore the average emissivity can be calculated as,

$$\epsilon = \frac{2(1-(1-\alpha))S}{4\sigma T^4} \tag{23}$$

In [ ]:
# Linear emissivity vs. CO2 concentration fit, eps = eps0 + m*conc

T1 = 286.7  # Avg surface temp from 1880-1900, K
eps1 = 2 * (1 - (1 - alpha) * S / (4 * sigma * T1**4)) # average emissivity from 1880-1900
c1 = (291 + 296) / 2  # Avg CO2 concentration from 1880-1900, ppm


T2 = 287.8  # Avg surface temp from 2020-2022, K
eps2 = 2 * (1 - (1 - alpha) * S / (4 * sigma * T2**4)) #emissivity in 2022
c2 = 418.56  # CO2 concentration in 2022, ppm

# Linear fit parameters
m = (eps2 - eps1) / (c2 - c1)  # Slope, 1/ppm
eps0 = eps1 - m * c1  # Intercept

The time step for the simulation will be one year and simualtion will be solved for 78 years from 2022 to 2100.

In [ ]:
# Timing
t0 = 0  # Initial time, s
dt = 365 * 24 * 3600  # Time step, s
K = 78  # Number of time steps
tf = t0 + K * dt  # Final time, s
t = np.arange(t0, tf + dt, dt)  # Time span, s
y = 2022 + t / (365 * 24 * 3600)  # Year

First we shall plot the emissivity and average surface temperature values against the $CO_2$ concentration as it varies between $200ppm$ and $500ppm$ using eqn $(21)$ and $(22)$ respectively.

In [ ]:
# CO2 Concentration Plot

# emissivity vs. CO2 concentration
c_plot = np.arange(200, 501) # CO2 concentration from 200 to 500 ppm
eps_values = eps0 + m * c_plot
temp_values = ((1 - alpha) * S / (4 * sigma * (1 - (eps0 + m * c_plot) / 2)))**(1 / 4)
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(c_plot, eps_values, label="Atmospheric emissivity", color="blue")
ax1.set_xlabel("Atmospheric CO$_2$ concentration (ppm)")
ax1.set_ylabel("Atmospheric emissivity", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")
ax1.set_ylim(0.6, 0.8)

#surface temperature vs. CO2 concentration
ax2 = ax1.twinx()
ax2.plot(c_plot, k2c(temp_values), label="Global average surface temperature", color="orange")
ax2.set_ylabel("Global average surface\ntemperature (°C)", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")
ax2.set_ylim(12.5, 16)  # Match MATLAB y-axis range for temperature

#annotation
lw = 2
ax1.axvline(c1, linestyle="--", color="black", linewidth=lw)
ax1.text(c1, 0.61, "1880–1900:\n294 ppm", rotation=90, verticalalignment="bottom", horizontalalignment="center")
ax1.axvline(c2, linestyle="--", color="black", linewidth=lw)
ax1.text(c2, 0.61, "2022:\n419 ppm", rotation=90, verticalalignment="bottom", horizontalalignment="center")
ax1.grid(visible=True, linestyle="--", linewidth=0.5)
fig.tight_layout()

Now that the input parameters and the control variable has been specified, we can now run the simulation using a classical non-linear ODE solver. First let us consider the nominal case of when the albedo is a constant and the emissivity increases linearly with time.

In [ ]:
# ==============================================================================
# Nominal Simulation
# ==============================================================================

# action (emissivity, from atmospheric CO2 concentration)
u_hat = eps + (410 - 315) / 60 * 0.05 / 280 * np.arange(1, K + 1)
#continuous-time disturbance (albedo and solar constant)
alpha_hat = alpha * np.ones(K)
wt_hat = (1 - alpha_hat) * S * np.pi * R**2 / C
# state
x0 = ((1 - alpha) * S / (4 * (1 - eps / 2) * sigma))**(1/4) # initial global average surface temperature, K
x_hat = nonlinear_climate_sim(t, x0, u_hat, wt_hat, beta) # global average surface temperature, K

Now let us consider the "true" case when the albedo varies with time. We will model the albedo variation as a sinusoidal function with a decadal period. Further, we will model the emissivity as linearly decreasing with time with a sinusoidal variation superimposed. The albedo and emissivity are the time dependent disturbance and control variables respectively in this case and assumed as inputs. These input profiles can be changed lead to different simulation results. They do not necessarily represent real world values.

In [ ]:
# ==============================================================================
# True Simulation
# ==============================================================================

# action (emissivity, from atmospheric CO2 concentration)
u = np.zeros(K)
u[0] = u_hat[0]
for k in range(K - 1):
    u[k + 1] = u[k] - 0.5 * (u_hat[k + 1] - u_hat[k])
u *= (1 + 0.01 * np.sin(2 * np.pi * y[:K] / 10))  # Perturbed albedo

# continuous-time disturbance (scaled albedo and solar constant)
alpha_t = alpha * (1 + 0.01 * np.sin(y[:K])) # Perturbed albedo
wt = (1 - alpha_t) * S * np.pi * R**2 / C
x = nonlinear_climate_sim(t, x0, u, wt, beta) # global average surface temperature, K


NameError: name 'np' is not defined

Finally we can solve the problem using the linearized solver developed previously.

In [ ]:
# ==============================================================================
# linearized simulation
# ==============================================================================

# Control perturbation
du = u - u_hat

# Continuous-time disturbance perturbation
dwt = wt - wt_hat

# Linearized state
x_lin = linearized_climate_sim(t, x_hat, u_hat, du, dwt, beta)  #  Global average surface temperature perturbation, K

# Convert x_lin to Celsius
x_lin_celsius = k2c(x_lin)

Now we shall plot the atmospheric emissivity calculated using the linear equation for constant and varying albedo's along with the surface temperature calculated from the non-linear as well as the linear solution.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 12))
fig.subplots_adjust(hspace=0.5)

# Second subplot: Atmospheric albedo
axs[0].step(y[:K], alpha_hat, linestyle="--", color="k", label="Nominal")
axs[0].step(y[:K], alpha_t, color="b", label="True")
axs[0].set_ylabel("Atmospheric\nalbedo\n$\\alpha(t)$")
axs[0].legend(loc="upper left")

# First subplot: Atmospheric emissivity
axs[1].step(y[:K], u_hat, linestyle="--", color="k", label="Nominal")
axs[1].step(y[:K], u, color="b", label="True")
axs[1].set_ylabel("Atmospheric\nemissivity\n$u(t)$")
axs[1].legend(loc="upper left")


# Third subplot: Surface temperature
axs[2].plot(y, k2c(x_hat), linestyle="--", color="k", label="Nominal")
axs[2].plot(y, k2c(x), color="b", label="True")
axs[2].plot(y, x_lin_celsius,
            marker="o", linestyle="none", color="magenta", label="Linearized")
axs[2].set_ylabel("Surface\ntemperature\n$x(t)$ (°C)")
axs[2].set_xlabel("Year")
axs[2].legend(loc="upper left")

Notice in the third plot the linearized solution does a good job of estimating the true non-linear solutionof surface temperature in that the points estimated by the linear solution are close to the true solution. The error between the two simulations can be plotted as,

In [ ]:
# Error plot
plt.figure(3, figsize=(10, 6))
plt.plot(y, x_lin_celsius - k2c(x), "k")  # Difference between linearized and true
plt.ylabel("Prediction error $x^{\\rm lin}(t) - x(t)$ ($^\circ$C)")
plt.xlabel("Year")
plt.xlim([y[0], y[-1]])
plt.grid(visible=True, linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.pause(0.001)
plt.show()


The error figure shows that as time progresses the error in the simulation due to linearization increases. The linear solution is consistently underpredicting the true temperature distribution as time progresses. However, for most practical purposes the linearized solution is a good approximation of the true non-linear solution. This is handy because linearized solutions are computationally less expensive and easier to work with especially when it comes to control and optimization problems.